In [1]:
from pathlib import Path
import cv2
import numpy as np
import pickle

# =========================
# AJUSTES
# =========================
CAM_INDEX = 0
ROI_W = 0.72          # ancho ROI respecto al frame
ROI_H = 0.72          # alto ROI respecto al frame
MIN_AREA = 1200       # sube si detecta basura
USE_TEMPLATES = True  # usa pills/shape_templates.pkl si existe

PKL_PATH = Path("pills/shape_templates.pkl")


# =========================
# ROI central
# =========================
def get_center_roi(frame, roi_w=ROI_W, roi_h=ROI_H):
    h, w = frame.shape[:2]
    rw = int(w * roi_w)
    rh = int(h * roi_h)
    x1 = (w - rw) // 2
    y1 = (h - rh) // 2
    x2 = x1 + rw
    y2 = y1 + rh
    return (x1, y1, x2, y2), frame[y1:y2, x1:x2]


# =========================
# Blanco del papel desde esquinas de la ROI
# =========================
def estimate_paper_bgr(roi, frac=0.10):
    h, w = roi.shape[:2]
    ph = max(12, int(h * frac))
    pw = max(12, int(w * frac))

    patches = [
        roi[0:ph, 0:pw],
        roi[0:ph, w-pw:w],
        roi[h-ph:h, 0:pw],
        roi[h-ph:h, w-pw:w]
    ]

    pixels = np.concatenate([p.reshape(-1, 3) for p in patches], axis=0)
    return np.median(pixels, axis=0).astype(np.uint8)


# =========================
# Máscara usando diferencia contra el blanco del papel
# =========================
def build_mask_white_bg(roi):
    paper_bgr = estimate_paper_bgr(roi)

    lab = cv2.cvtColor(roi, cv2.COLOR_BGR2LAB).astype(np.float32)
    paper_lab = cv2.cvtColor(
        np.uint8([[paper_bgr]]), cv2.COLOR_BGR2LAB
    )[0, 0].astype(np.float32)

    # Distancia al color del papel
    delta = np.linalg.norm(lab - paper_lab, axis=2)
    delta = cv2.normalize(delta, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    delta = cv2.GaussianBlur(delta, (5, 5), 0)

    _, mask = cv2.threshold(delta, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    return mask, paper_bgr, delta


# =========================
# Contornos válidos
# =========================
def get_valid_contours(mask, min_area=MIN_AREA):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    h, w = mask.shape[:2]
    valid = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue

        x, y, cw, ch = cv2.boundingRect(cnt)

        # ignorar objetos pegados al borde
        if x <= 2 or y <= 2 or (x + cw) >= (w - 2) or (y + ch) >= (h - 2):
            continue

        valid.append(cnt)

    valid.sort(key=cv2.contourArea, reverse=True)
    return valid


# =========================
# Forma
# =========================
def classify_shape_rules(cnt):
    area = cv2.contourArea(cnt)
    peri = cv2.arcLength(cnt, True)
    if peri == 0:
        return "unknown", {}

    circularity = 4 * np.pi * area / (peri * peri)

    rect = cv2.minAreaRect(cnt)
    (_, _), (w, h), _ = rect
    w = max(w, 1e-6)
    h = max(h, 1e-6)
    aspect = max(w, h) / min(w, h)

    info = {
        "area": area,
        "peri": peri,
        "circularity": circularity,
        "aspect": aspect
    }

    if circularity >= 0.80 and aspect <= 1.18:
        return "circle", info

    if aspect >= 2.0:
        return "pill", info

    if aspect > 1.15:
        return "caplet", info

    return "unknown", info


def load_templates():
    if USE_TEMPLATES and PKL_PATH.exists():
        with open(PKL_PATH, "rb") as f:
            return pickle.load(f)
    return None


def classify_shape(cnt, templates):
    if not templates:
        return classify_shape_rules(cnt)

    best_label = "unknown"
    best_score = float("inf")

    for label, tmpl in templates.items():
        score = cv2.matchShapes(cnt, tmpl, cv2.CONTOURS_MATCH_I1, 0.0)
        if score < best_score:
            best_score = score
            best_label = label

    label_rules, info = classify_shape_rules(cnt)
    info["match_score"] = best_score

    if best_score > 0.20:
        return label_rules, info

    return best_label, info


# =========================
# Color
# =========================
def classify_color(roi, cnt):
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    obj_mask = np.zeros(roi.shape[:2], dtype=np.uint8)
    cv2.drawContours(obj_mask, [cnt], -1, 255, -1)

    # medir color solo en el interior
    inner = cv2.erode(obj_mask, np.ones((7, 7), np.uint8), iterations=1)
    if cv2.countNonZero(inner) < 30:
        inner = obj_mask

    pixels = hsv[inner == 255]
    if len(pixels) == 0:
        return "unknown", {}

    # quitar reflejos muy fuertes
    pixels = pixels[(pixels[:, 2] < 245)]
    if len(pixels) == 0:
        pixels = hsv[obj_mask == 255]

    h = float(np.median(pixels[:, 0]))
    s = float(np.median(pixels[:, 1]))
    v = float(np.median(pixels[:, 2]))

    info = {"H": h, "S": s, "V": v}

    # neutros
    if v < 55:
        return "black", info

    if s < 30:
        if v < 150:
            return "gray", info
        return "cream", info

    # colores vivos
    if h <= 8 or h >= 172:
        if v > 140 and s < 140:
            return "pink", info
        return "red", info

    if 9 <= h <= 20:
        return "orange", info

    if 21 <= h <= 34:
        return "yellow", info

    if 35 <= h <= 85:
        return "green", info

    if 86 <= h <= 135:
        return "blue", info

    if 136 <= h <= 171:
        return "pink", info

    return "unknown", info


# =========================
# Cámara
# =========================
templates = load_templates()
if templates:
    print("Templates cargados:", list(templates.keys()))
else:
    print("Sin templates: usando reglas geométricas.")

cap = cv2.VideoCapture(CAM_INDEX)
if not cap.isOpened():
    cap = cv2.VideoCapture(CAM_INDEX, cv2.CAP_DSHOW)

print("q = salir")

while True:
    ok, frame = cap.read()
    if not ok:
        print("No se pudo leer la cámara.")
        break

    display = frame.copy()

    (x1, y1, x2, y2), roi = get_center_roi(frame)
    mask, paper_bgr, delta = build_mask_white_bg(roi)
    contours = get_valid_contours(mask)

    roi_vis = roi.copy()

    cv2.rectangle(display, (x1, y1), (x2, y2), (0, 255, 255), 2)
    cv2.putText(display, "ROI", (x1, max(25, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    for cnt in contours:
        shape_label, shape_info = classify_shape(cnt, templates)
        color_label, color_info = classify_color(roi, cnt)

        x, y, w, h = cv2.boundingRect(cnt)

        cv2.drawContours(roi_vis, [cnt], -1, (0, 255, 0), 2)
        cv2.rectangle(roi_vis, (x, y), (x + w, y + h), (255, 0, 0), 2)

        label = f"{color_label} | {shape_label}"
        cv2.putText(
            roi_vis, label, (x, max(20, y - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2
        )

        hsv_txt = f"H:{color_info.get('H', 0):.0f} S:{color_info.get('S', 0):.0f} V:{color_info.get('V', 0):.0f}"
        cv2.putText(
            roi_vis, hsv_txt, (x, y + h + 18),
            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1
        )

    paper_box = np.full((40, 120, 3), paper_bgr, dtype=np.uint8)
    cv2.imshow("paper_ref", paper_box)
    cv2.imshow("camera", display)
    cv2.imshow("delta_white", delta)
    cv2.imshow("mask", mask)
    cv2.imshow("roi_result", roi_vis)

    key = cv2.waitKey(1) & 0xFF
    if key == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

Sin templates: usando reglas geométricas.
q = salir
